# `ov_utils` — benchmark helpers

This notebook documents **`benchmark_two_stage_per_device`**, **`print_two_stage_benchmark_table`**, and **`plot_benchmark_det_cls`** from [`ov_utils.py`](ov_utils.py). Run this folder as the kernel working directory so `import ov_utils` works.

The main lab notebook keeps only **`BENCHMARK_PRECISION`**, paths, and `core` / `image_bgr` / `THRESHOLD`; the heavy logic lives here in Python.

**Fair timing:** detection is compiled once on **`shared_detection_device`** (default GPU if available, else CPU); each row only changes the **classification** device. Default benchmark rows are **CPU** and **GPU** only (pass ``devices`` to add NPU). One untimed warmup run stabilizes the shared detector before measurements.


In [ ]:
import os
from pathlib import Path

import cv2
import openvino as ov

from ov_utils import (
    benchmark_two_stage_per_device,
    plot_benchmark_det_cls,
    print_two_stage_benchmark_table,
)


## Inputs (edit to match your tree)

- **`BASE`** — deployment folder containing `models/`
- **`BENCHMARK_PRECISION`** — e.g. `INT8`
- **`image_bgr`** — sample image for timing


In [ ]:
BASE = Path(os.environ.get("OV_DEPLOY_BASE", os.environ.get("GETI_DEPLOY_BASE", "."))).resolve()
MODELS = BASE / "models"
THRESHOLD = 0.225
BENCHMARK_PRECISION = "INT8"

detection_xml = MODELS / BENCHMARK_PRECISION / "Detection" / "model.xml"
classification_xml = MODELS / BENCHMARK_PRECISION / "Classification" / "model.xml"
sample = BASE / "media" / "sample_image_1.jpg"
if not sample.is_file():
    sample = next(MODELS.rglob("*.jpg"), None)
if sample is None or not sample.is_file():
    raise FileNotFoundError("Add a .jpg under media/ or models/ for smoke test")
image_bgr = cv2.imread(str(sample))

core = ov.Core()
print("devices:", core.available_devices)
print("det:", detection_xml.is_file(), "cls:", classification_xml.is_file())


In [ ]:
if not detection_xml.is_file() or not classification_xml.is_file():
    raise FileNotFoundError(f"Need IRs under models/{BENCHMARK_PRECISION}/")

rows = benchmark_two_stage_per_device(
    core,
    detection_xml,
    classification_xml,
    image_bgr,
    THRESHOLD,
)
print_two_stage_benchmark_table(rows, BENCHMARK_PRECISION)
plot_benchmark_det_cls(rows, BENCHMARK_PRECISION)
